# Protein Function Prediction – Data Preparation
**Project:** COMP 3608 B‑rank mission  
**This notebook:** downloads Kaggle datasets → `data/raw/`, cleans & merges them, engineers features, and saves processed data to `data/processed/`.

In [1]:
# 0. Environment
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')

# Create directory structure
RAW_DIR = Path('data/raw')
PROC_DIR = Path('data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
!pip install kagglehub -q

In [3]:
# 1. Download datasets to data/raw/
import kagglehub

# GO annotations
path_go = kagglehub.dataset_download(
    "nikitamanaenkov/protein-sequences-with-go-annotations",
    output_dir=str(RAW_DIR / 'go_annotations')
)
# Simulated 1
path_sim1 = kagglehub.dataset_download(
    "willianoliveiragibin/bioinformatics-simulated",
    output_dir=str(RAW_DIR / 'simulated_1')
)

# Human protein sequences
path_human = kagglehub.dataset_download(
    "usamaraheem/human-protein-sequences-and-function-annotations",
    output_dir=str(RAW_DIR / 'human-protein-sequences')
)

print("Datasets downloaded to:", str(RAW_DIR))

Datasets downloaded to: data/raw


In [4]:
import re
import pandas as pd
from collections import Counter
from pathlib import Path

RAW_DIR = Path("data/raw")

# GO term to function class mapping 
# Maps GO terms to the 5 bioinformatics class labels
GO_TO_CLASS = {
    # Enzyme — catalytic activity
    "GO:0003824": "Enzyme",   # catalytic activity
    "GO:0016301": "Enzyme",   # kinase activity
    "GO:0016787": "Enzyme",   # hydrolase activity
    "GO:0016740": "Enzyme",   # transferase activity
    "GO:0016829": "Enzyme",   # lyase activity
    "GO:0016853": "Enzyme",   # isomerase activity
    "GO:0016874": "Enzyme",   # ligase activity
    "GO:0016491": "Enzyme",   # oxidoreductase activity

    # Receptor — signal reception
    "GO:0004872": "Receptor", # receptor activity
    "GO:0038023": "Receptor", # signaling receptor activity
    "GO:0004888": "Receptor", # transmembrane signaling receptor
    "GO:0004930": "Receptor", # G protein-coupled receptor
    "GO:0004984": "Receptor", # olfactory receptor activity
    "GO:0004896": "Receptor", # cytokine receptor activity

    # Transporter — molecule transport
    "GO:0005215": "Transporter", # transporter activity
    "GO:0022857": "Transporter", # transmembrane transporter
    "GO:0015075": "Transporter", # ion transmembrane transporter
    "GO:0005344": "Transporter", # oxygen carrier activity
    "GO:0022890": "Transporter", # inorganic cation transmembrane transporter

    # Structural — structural roles
    "GO:0005198": "Structural",  # structural molecule activity
    "GO:0003779": "Structural",  # actin binding
    "GO:0005200": "Structural",  # structural constituent of cytoskeleton
    "GO:0003735": "Structural",  # structural constituent of ribosome
    "GO:0042302": "Structural",  # structural constituent of cuticle
}

def assign_class(go_terms):
    """
    Assign one of the 5 bioinformatics classes to a protein
    based on its GO terms. Falls back to 'Others' if no match.
    """
    counts = Counter()
    for term in go_terms:
        cls = GO_TO_CLASS.get(term)
        if cls:
            counts[cls] += 1
    if counts:
        return counts.most_common(1)[0][0]
    return "Others"

# Parse FASTA 
def parse_fasta(fasta_path):
    sequences  = {}
    current_id = None
    current_seq = []
    with open(fasta_path, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            line = line.strip()
            if line.startswith(">"):
                if current_id:
                    sequences[current_id] = "".join(current_seq)
                match      = re.search(r"\|(.*?)\|", line)
                current_id = match.group(1) if match else line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
    if current_id:
        sequences[current_id] = "".join(current_seq)
    return sequences

# Parse annotations → extract GO terms
def parse_annotations(annot_path):
    go_map = {}
    with open(annot_path, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            protein_id = line.split(maxsplit=1)[0]
            terms      = re.findall(r"(GO:\d+)", line)
            if terms:
                go_map.setdefault(protein_id, []).extend(terms)
    return go_map

# Build dataframe 
fasta_path = RAW_DIR / "human-protein-sequences/ALL-HUMAN-0001 SEQUENCES.fasta"
annot_path = RAW_DIR / "human-protein-sequences/ALL-HUMAN-0001-ANNOTATIONS.txt"

sequences = parse_fasta(str(fasta_path))
go_map    = parse_annotations(str(annot_path))

valid_ids = list(set(sequences) & set(go_map))

df_human = pd.DataFrame({
    "sequence": [sequences[i] for i in valid_ids],
    "label":    [assign_class(go_map[i])   for i in valid_ids],
})

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")
df_human["sequence"] = df_human["sequence"].apply(
    lambda s: "".join([aa if aa in VALID_AA else "X" for aa in str(s).upper()])
)
df_human = df_human[df_human["sequence"].str.len().between(10, 1000)]
df_human.drop_duplicates(subset="sequence", inplace=True)
df_human = df_human[df_human["label"].notna()].reset_index(drop=True)

print(f"Shape : {df_human.shape}")
print(f"\nLabel distribution:")
print(df_human["label"].value_counts())
print(f"\nSample:")
print(df_human[["sequence", "label"]].head())

Shape : (27954, 2)

Label distribution:
label
Others         24055
Enzyme          1402
Receptor        1130
Structural       973
Transporter      394
Name: count, dtype: int64

Sample:
                                            sequence   label
0  ISLEHEILLHPRYFGPNLLNTVKQKLFTEVEGTCTGKIKTVWGRPE...  Others
1  XDGLHNMGGDPITVIDEIRDLLYIGKDRKNPREDYLDVYVFGVGPL...  Others
2  MDYLLMIFSLLFVACQGAPETAVLGAELSAVGENGGEKPTPSPPWR...  Others
3  MKMKKFQIPVSFQDLTVNFTQEEWQQLDPAQRLLYRDVMLENYSNL...  Others
4  EKFKKAQELGATECLNPQDLKKPIQEVLFDMTDAGIDFCFEAIGNL...  Enzyme


In [5]:
# 2. Load & explore
def find_csv(directory):
    return list(Path(directory).rglob('*.csv'))

df_go = pd.read_csv(find_csv(path_go)[0])
df_sim1 = pd.read_csv(find_csv(path_sim1)[0])

print("GO columns:", df_go.columns.tolist())
print("Sim1 columns:", df_sim1.columns.tolist())
print("human protein columns:", df_human.columns.tolist())

GO columns: ['Entry', 'Sequence', 'GO_list', 'GO', 'GO_id', 'GO_namespace', 'GO_parents', 'GO_children', 'seq_length', 'mol_weight', 'pI', 'gravy', 'instability', 'aromaticity', 'helix', 'turn', 'sheet', 'aa_A', 'aa_C', 'aa_D', 'aa_E', 'aa_F', 'aa_G', 'aa_H', 'aa_I', 'aa_K', 'aa_L', 'aa_M', 'aa_N', 'aa_P', 'aa_Q', 'aa_R', 'aa_S', 'aa_T', 'aa_V', 'aa_W', 'aa_Y']
Sim1 columns: ['ID_Proteína', 'Sequência', 'Massa_Molecular', 'Ponto_Isoelétrico', 'Hidrofobicidade', 'Carga_Total', 'Proporção_Polar', 'Proporção_Apolar', 'Comprimento_Sequência', 'Classe']
human protein columns: ['sequence', 'label']


In [6]:
df_sim1 = df_sim1.rename(columns={
    "ID_Proteína": "Protein_ID",
    "Sequência": "Sequence",
    "Massa_Molecular": "Molecular_Weight",
    "Ponto_Isoelétrico": "Isoelectric_Point",
    "Hidrofobicidade": "Hydrophobicity",
    "Carga_Total": "Net_Charge",
    "Proporção_Polar": "Polar_Ratio",
    "Proporção_Apolar": "NonPolar_Ratio",
    "Comprimento_Sequência": "Sequence_Length",
    "Classe": "Class"
})

In [10]:
df_sim1["Class"] = df_sim1["Class"].replace({
    "Estrutural": "Structural",
    "Receptora": "Receptor",
    "Enzima" : "Enzyme",
    "Transporte" : "Transporter",
    "Outras" : "Others",
})

In [11]:
# 3. Standardise columns to 'sequence' and 'label_raw'
def standardise_columns(df):
    """Auto‑detect and rename sequence + label columns."""
    df = df.copy()
    # Sequence column keywords
    seq_keywords = ['seq', 'Sequence']
    # Label column keywords (including GO-specific)
    label_keywords = ['go', 'label', 'function', 'class']
    
    seq_cols = [c for c in df.columns if any(k in c.lower() for k in seq_keywords)]
    if not seq_cols:
        raise KeyError(f"No sequence column found in {list(df.columns)}")
    seq_col = seq_cols[0]
    
    label_cols = [c for c in df.columns if any(k in c.lower() for k in label_keywords)]
    if not label_cols:
        raise KeyError(f"No label column found in {list(df.columns)}")
    label_col = label_cols[0]
    
    print(f"Auto‑detected: seq='{seq_col}', label='{label_col}'")
    df.rename(columns={seq_col: 'sequence', label_col: 'label_raw'}, inplace=True)
    return df[['sequence', 'label_raw']]

df_go = standardise_columns(df_go)
df_sim1 = standardise_columns(df_sim1)
df_human = standardise_columns(df_human)

print("\nSample GO:")
print(df_go.head(2))

Auto‑detected: seq='Sequence', label='GO_list'
Auto‑detected: seq='Sequence', label='Class'
Auto‑detected: seq='sequence', label='label'

Sample GO:
                                            sequence  \
0  MNIDMNWLGQLLGSDWEIFPAGGATGDAYYAKHNGQQLFLKRNSSP...   
1  MFKKHTISLLIIFLLASAVLAKPIEAHTVSPVNPNAQQTTKTVMNW...   

                                           label_raw  
0  ['cytoplasm [GO:0005737]', 'ATP binding [GO:00...  
1  ['extracellular region [GO:0005576]', 'mannan ...  


In [12]:
# 4. Map GO terms to high‑level functional categories
def map_go(go_str):
    if pd.isna(go_str): return 'others'
    s = str(go_str)
    if any(t in s for t in ['catalytic','hydrolase','kinase','transferase','enzyme']):
        return 'enzyme'
    if 'binding' in s or 'receptor' in s:
        return 'binding'
    if 'transporter' in s:
        return 'transporter'
    if 'signal' in s or 'transducer' in s:
        return 'signal_transduction'
    if 'structural' in s or 'cytoskeleton' in s:
        return 'structural'
    return 'others'

df_go['label'] = df_go['label_raw'].apply(map_go)

def clean_label(raw):
    return 'others' if pd.isna(raw) else str(raw).strip().lower().replace(' ', '_')

df_sim1['label'] = df_sim1['label_raw'].apply(clean_label)
# df_sim2['label'] = df_sim2['label_raw'].apply(clean_label)

for d in [df_go, df_sim1]:
    d.drop(columns='label_raw', inplace=True)

In [13]:
# 5. Merge & clean sequences
df_all = pd.concat([df_go, df_sim1, df_human], ignore_index=True)
df_all.dropna(subset=['sequence','label'], inplace=True)

VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')
df_all['sequence'] = df_all['sequence'].apply(
    lambda s: ''.join([aa if aa in VALID_AA else 'X' for aa in str(s).upper()])
)
df_all = df_all[df_all['sequence'].str.len().between(10, 1000)]
df_all.drop_duplicates(subset='sequence', inplace=True)

# Keep classes with ≥5 samples
vc = df_all['label'].value_counts()
df_all = df_all[df_all['label'].isin(vc[vc >= 5].index)]

print("Final shape:", df_all.shape)
print("Label distribution:")
print(df_all['label'].value_counts())

Final shape: (16159, 3)
Label distribution:
label
enzyme         3266
structural     3233
transporter    3233
others         3199
receptor       3125
binding         103
Name: count, dtype: int64


In [14]:
# 6. Save processed sequences (for CNN) to data/processed/
df_all.to_csv(PROC_DIR / 'processed_sequences.csv', index=False)

# Save max sequence length (95th percentile) for padding
max_len = int(np.percentile(df_all['sequence'].apply(len), 95))
np.save(PROC_DIR / 'max_len.npy', max_len)
print("Max length for padding:", max_len)

Max length for padding: 290


In [15]:
# 7. Feature engineering for classical models (save to data/processed/)
AA_ORDER = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa:i for i,aa in enumerate(AA_ORDER)}

def aac(seq):
    counts = np.zeros(20)
    for aa in seq:
        if aa in AA_TO_IDX:
            counts[AA_TO_IDX[aa]] += 1
    return counts / len(seq) if len(seq)>0 else counts

def dpc(seq):
    vec = np.zeros(400)
    if len(seq)<2: return vec
    for i in range(len(seq)-1):
        di = seq[i:i+2]
        if di[0] in AA_TO_IDX and di[1] in AA_TO_IDX:
            idx = AA_TO_IDX[di[0]]*20 + AA_TO_IDX[di[1]]
            vec[idx] += 1
    return vec / (len(seq)-1)

X_aac = np.array([aac(seq) for seq in df_all['sequence']])
X_dpc = np.array([dpc(seq) for seq in df_all['sequence']])

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df_all['label'])
class_names = le.classes_

# Save features
np.savez(PROC_DIR / 'features.npz', X_aac=X_aac, X_dpc=X_dpc, y=y)
np.save(PROC_DIR / 'class_names.npy', class_names)
print("Features saved: AAC", X_aac.shape, "DPC", X_dpc.shape, "classes", len(class_names))

Features saved: AAC (16159, 20) DPC (16159, 400) classes 6


---
**Data preparation complete.** All processed artifacts are now in `data/processed/`.